# Kachel 02 — Konzentration der Hamburger Zuwendungen

**Frage:** Welcher Anteil der Empfänger bekommt die Hälfte des Geldes?

Dieses Notebook ist der Weg zum Befund. Drei Eigenheiten der Quelle mussten
unterwegs entschärft werden — sie sind hier einzeln nachgewiesen, weil jede
einzelne das Ergebnis um ein Vielfaches verfälscht hätte.

In [ ]:
import pandas as pd

from core.fetch import zuwendungsvorgaenge

dateien = zuwendungsvorgaenge(2024)
dateien

## Falle 1: Die Quartalsdateien sind kumulativ

Die Stadt veröffentlicht vier Dateien pro Jahr. Das legt nahe, sie zu
einem Jahr zusammenzuhängen — genau das wäre falsch. Jede Datei ist ein
vollständiger Auszug der letzten rund zehn Jahre.

In [ ]:
for datei in dateien:
    df = pd.read_excel(datei, sheet_name="Zuwendungsbescheide")
    datum = pd.to_datetime(df["Bescheiddatum"], errors="coerce")
    print(f"{datei.name}: {len(df):>6} Zeilen, {datum.min():%Y-%m-%d} bis {datum.max():%Y-%m-%d}")

Alle vier decken denselben Zeitraum ab, nur unterschiedlich weit fortgeschrieben.
Es genügt also die Datei zu Q4 — die vier zu verketten würde fast alles vierfach zählen.

In [ ]:
df = pd.read_excel(dateien[-1], sheet_name="Zuwendungsbescheide")
df["Bescheiddatum"] = pd.to_datetime(df["Bescheiddatum"], errors="coerce")
print(list(df.columns))
df["Bescheidart"].value_counts()

## Falle 2: Ein Vorgang hat viele Bescheide — und jeder nennt den vollen Betrag

Die `INEZ-Nummer` identifiziert den Vorgang. Rund die Hälfte aller Vorgänge hat
mehr als einen Bescheid, der Spitzenwert liegt bei 23. Entscheidend ist:
`Zuwendungssumme` ist keine Veränderung, sondern der jeweils **aktuelle
Gesamtbetrag** des Vorgangs.

In [ ]:
haeufigkeit = df["INEZ-Nummer"].value_counts()
print(f"Vorgänge: {df['INEZ-Nummer'].nunique()}")
print(f"davon mit mehr als einem Bescheid: {(haeufigkeit > 1).sum()}")
print(f"meiste Bescheide auf einem Vorgang: {haeufigkeit.max()}")

beispiel = haeufigkeit.index[0]
df[df["INEZ-Nummer"] == beispiel].sort_values("Bescheiddatum")[
    ["Bescheiddatum", "Bescheidart", "Zuwendungssumme", "Zuwendungsempfänger"]
]

Der Betrag wandert über die Jahre auf und ab — er wird korrigiert, nicht addiert.
Die Zeilen zu summieren, hätte die Gesamtsumme vielfach überhöht.

**Konsequenz:** je Vorgang den jüngsten Bescheid nehmen. Ablehnungen und
Aufhebungen fallen dabei heraus, weil sie keinen ausgezahlten Betrag begründen.
Das Jahr eines Vorgangs ist das Jahr seines *ersten* Bescheids — also das
Bewilligungsjahr. Beides steckt in `core.analyse.lade_zuwendungsvorgaenge`.

## Falle 3: Keine Ortsangabe

Der Projektplan hielt sich offen, ob "Zuwendungen × Sozialmonitoring" als
räumliche Kachel taugt. Die Spaltenliste oben beantwortet das: es gibt weder
Bezirk noch Adresse noch Stadtteil. Ein räumlicher Join ist mit diesem
Datensatz nicht möglich — Kachel 04 bleibt beim Baumkataster.

## Falle 4: Die Spitze der Rangliste besteht nicht aus Vereinen

Der Projektplan warnt vor Durchleitungsstellen. Die Prüfung zeigt etwas
Verwandtes: ganz oben stehen städtische Eigenbetriebe, Staatstheater,
Landesmuseen und Forschungsorganisationen. Die Stadt fördert überwiegend
sich selbst.

In [ ]:
from core.analyse import lade_zuwendungsvorgaenge
from core.traeger import klassifiziere

vorgaenge = lade_zuwendungsvorgaenge(2024)
je_empfaenger = (
    vorgaenge.assign(empfaenger=vorgaenge["Zuwendungsempfänger"].str.strip())
    .groupby("empfaenger")["Zuwendungssumme"]
    .sum()
    .sort_values(ascending=False)
)
top = je_empfaenger.head(15).to_frame("summe")
top["gruppe"] = top.index.map(klassifiziere)
top["summe"] = (top["summe"] / 1e6).round(1)
top

Ohne diese Trennung läse sich der Befund als "ein paar Vereine bekommen fast
alles". Die Einordnung steckt in `core.traeger`: die 100 größten Empfänger
(82 % der Summe) sind einzeln geprüft, der Rest läuft über Rechtsformregeln
mit "frei" als konservativer Voreinstellung.

## Der Befund

In [ ]:
from core.analyse import zuwendungskonzentration

befund = zuwendungskonzentration(2024)
for gruppe, werte in befund.kennzahlen.items():
    print(gruppe, werte)

Beide Gruppen sind für sich konzentriert, aber unterschiedlich stark: unter den
freien Trägern ist die Ungleichheit *größer* (Gini 0,85 gegen 0,70). Die
öffentliche Seite ist ein kleiner Kreis großer Empfänger, die freie Seite eine
lange Reihe kleiner Vereine mit wenigen großen Wohlfahrtsverbänden an der Spitze.

In [ ]:
from IPython.display import Image

from core.render import render_tile

pfad = render_tile(befund, "kachel_02_zuwendungen.png")
Image(filename=str(pfad))